# 04 — DINOv2 ile sızıntı temizliği (Faz 4a)

**Çalışma zamanı: GPU önerilir (ücretsiz T4 yeterli).** Runtime → Change runtime type → T4 GPU. GPU'da toplam ~20 dk; GPU yoksa CPU'da da çalışır, embedding adımı ~1 saat sürer.

**Neden:** 03'te phash, 8 bitten sonra farklı sahneleri de "benzer" saymaya başladı. Bu yüzden FASDD testinde 10–12 bit farkla kalan aynı-sahne kareleri ayıramadı (örneklenen 8 çiftin 4'ü aynı sahneydi). DINOv2, görüntünün *ne gösterdiğini* temsil eden bir vektör üretir ve aynı sahneyi benzer görünen farklı bir sahneden phash'ten çok daha iyi ayırır.

**Yöntem**
1. Her görüntü için DINOv2 vektörü hesaplanır (kosinüs benzerliği, 1 = aynı).
2. Eşik veriyle ayarlanır:
   - **Kesin farklı referansı:** Heinola karelerinin diğer Boreal yangınlarına en yüksek benzerliği. Aynı drone, benzer orman, ama farklı yangın. Zorlayıcı bir referans.
   - **Kesin aynı referansı:** FASDD'de aynı phash grubundaki en yakın kare.
3. Eşiğin üstündeki FASDD çiftleri aynı gruba bağlanır, gruplar 03'teki kurallarla yeniden dağıtılır.
4. Sızıntı tekrar ölçülür, dosyalar taşınır, `firesmoke_v2.zip` Drive'a yazılır. **v1 silinmez**, yedek olarak kalır.

Boreal'e dokunulmaz: olay ve video bazlı ayrım zaten kesin, v1'deki split'i aynen korunur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L || echo 'GPU yok — CPU ile devam (embedding ~1 saat)'

In [ ]:
import os, json, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as TF

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
V1_ZIP = ROOT / 'datasets' / 'firesmoke_v1.zip'
V2_ZIP = ROOT / 'datasets' / 'firesmoke_v2.zip'
WORK   = Path('/content/firesmoke_v1')        # v1 buraya açılır, sonunda v2 olarak yeniden adlandırılır
DATA   = Path('/content/firesmoke_v2')
EMB    = REP / 'dinov2s_emb.npz'               # embedding önbelleği (tekrar çalıştırmada yeniden hesaplanmaz)

# --- Ayarlar (03 ile aynı dağıtım kuralları) ---------------------------------
SEED       = 0
RATIOS     = {'train': 0.70, 'val': 0.15, 'test': 0.15}
BOREAL_VAL = 0.15
TEST_EVENT = 'heinola'
BIG_GROUP  = 300
CATS       = ['fire+smoke', 'sadece fire', 'sadece smoke', 'negatif']
NEG_Q      = 0.99     # eşik = "kesin farklı" benzerliklerinin %99'luk değeri
TAU        = None     # None: otomatik. Gözle kontrolden sonra elle girilebilir (örn. 0.85)
K          = 64       # her görüntü için bakılan en yakın komşu sayısı
PRETRAINED = True
# ------------------------------------------------------------------------------

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

## 1. v1'i aç

In [ ]:
t0 = time.time()
if not (WORK / 'manifest.csv').exists():
    assert V1_ZIP.exists(), f'{V1_ZIP} yok — önce 03 çalıştırılmalı'
    subprocess.run(['unzip', '-q', '-o', str(V1_ZIP), '-d', str(WORK.parent)], check=True)
m = pd.read_csv(WORK / 'manifest.csv')
missing = int((~m.file.map(lambda f: (WORK / f).exists())).sum())
print(f'{len(m)} görüntü, eksik dosya: {missing}  ({time.time() - t0:.0f} sn)')
assert missing == 0
display(m.groupby(['source', 'split']).size().unstack())

## 2. DINOv2 vektörleri

Model: `dinov2_vits14` (küçük sürüm, 384 boyutlu vektör). Görüntüler 224×224'e küçültülür. Sonuç Drive'a önbelleklenir; notebook tekrar çalışırsa bu adım atlanır.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

class Imgs(torch.utils.data.Dataset):
    def __init__(self, files):
        self.files = list(files)
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        with Image.open(WORK / self.files[i]) as im:
            im.draft('RGB', (448, 448))
            im = im.convert('RGB').resize((224, 224), Image.BILINEAR)
        x = (np.asarray(im, np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x.transpose(2, 0, 1))

cached = None
if EMB.exists():
    z = np.load(EMB, allow_pickle=False)
    if len(z['uid']) == len(m) and (z['uid'] == m.uid.to_numpy().astype(str)).all():
        cached = z['emb']
if cached is not None:
    emb = cached
    print('önbellekten yüklendi:', emb.shape)
else:
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=PRETRAINED,
                           trust_repo=True).to(DEV).eval()
    if DEV == 'cuda':
        model = model.half()
    dl = torch.utils.data.DataLoader(Imgs(m.file), batch_size=128 if DEV == 'cuda' else 32,
                                     num_workers=os.cpu_count() or 2, pin_memory=(DEV == 'cuda'))
    out, t0 = [], time.time()
    with torch.inference_mode():
        for b, x in enumerate(dl, 1):
            x = x.to(DEV, non_blocking=True)
            f = model(x.half() if DEV == 'cuda' else x)
            out.append(TF.normalize(f.float(), dim=1).cpu().numpy().astype(np.float16))
            if b % 20 == 0:
                done = min(b * dl.batch_size, len(m))
                print(f'{done}/{len(m)}  {time.time() - t0:.0f} sn')
    emb = np.concatenate(out)
    np.savez(EMB, emb=emb, uid=m.uid.to_numpy().astype(str))
    print(f'hesaplandı: {emb.shape}, {time.time() - t0:.0f} sn  →  {EMB}')

## 3. Eşiği veriyle ayarla

Üç dağılım çizilir; her biri bir görüntünün kendi referans kümesindeki **en yüksek** benzerliğidir:
- **Kesin farklı** (kırmızı): Heinola karesi ↔ diğer Boreal yangınları.
- **Kesin aynı** (yeşil): FASDD karesi ↔ aynı phash grubundaki en yakın kare.
- **Test** (mavi): FASDD test karesi ↔ FASDD train/val.

Otomatik eşik, kırmızı dağılımın %99'luk değeri: farklı yangınların sadece %1'i bu eşiği geçiyor. Mavinin eşiği geçen kısmı, sızıntı şüphesi olan test görüntüleridir.

In [ ]:
E = torch.from_numpy(np.array(emb)).to(DEV)
E = E.half() if DEV == 'cuda' else E.float()

def topk_sim(qi, bi, k=1, chunk=2048):
    """qi'deki her görüntü için bi içindeki en benzer k görüntü: (benzerlik, indeks)."""
    qi, bi = np.array(qi), np.array(bi)          # kopya: torch salt-okunur diziyi indeks olarak sevmiyor
    k = min(k, len(bi))
    Eb, vals, idxs = E[bi], [], []
    for s in range(0, len(qi), chunk):
        sim = E[qi[s:s + chunk]] @ Eb.T
        v, ix = sim.float().topk(k, dim=1)
        vals.append(v.cpu().numpy()); idxs.append(bi[ix.cpu().numpy()])
    if not vals:
        return np.zeros((0, k)), np.zeros((0, k), dtype=int)
    return np.concatenate(vals), np.concatenate(idxs)

def sel(mask):
    return m.index[mask].to_numpy()

is_f, is_b = m.source == 'fasdd', m.source == 'boreal'
hein  = sel(is_b & (m.event == TEST_EVENT))
other = sel(is_b & (m.event != TEST_EVENT))
d_neg, _ = topk_sim(hein, other)
d_neg = d_neg[:, 0]

# kesin aynı: FASDD grubundaki en yakın diğer kare
d_pos = []
for _, g in m[is_f].groupby('group'):
    ix = g.index.to_numpy()
    if len(ix) < 2:
        continue
    v, _ = topk_sim(ix, ix, k=2)
    d_pos.append(v[:, 1])
d_pos = np.concatenate(d_pos)

f_test = sel(is_f & (m.split == 'test'))
f_rest = sel(is_f & (m.split != 'test'))
d_test, n_test = topk_sim(f_test, f_rest)
d_test, n_test = d_test[:, 0], n_test[:, 0]

tau = float(np.quantile(d_neg, NEG_Q)) if TAU is None else float(TAU)
print(f'eşik τ = {tau:.3f}  ({"otomatik" if TAU is None else "elle"})')
print(f'kesin farklı ≥ τ : %{100 * (d_neg >= tau).mean():.1f}  (n={len(d_neg)})')
print(f'kesin aynı   ≥ τ : %{100 * (d_pos >= tau).mean():.1f}  (n={len(d_pos)})  ← aynı sahneyi yakalama oranı')
print(f'FASDD test   ≥ τ : %{100 * (d_test >= tau).mean():.1f}  ({int((d_test >= tau).sum())} / {len(d_test)} görüntü)')

bins = np.linspace(min(d_neg.min(), d_test.min(), d_pos.min()), 1, 60)
plt.figure(figsize=(9, 3.5))
for d, c, lab in [(d_neg, 'tab:red', 'kesin farklı (Heinola ↔ diğer yangınlar)'),
                  (d_pos, 'tab:green', 'kesin aynı (FASDD grup içi)'),
                  (d_test, 'tab:blue', 'FASDD test ↔ train/val')]:
    plt.hist(d, bins=bins, density=True, alpha=.45, color=c, label=lab)
plt.axvline(tau, color='k', ls='--', label=f'τ = {tau:.3f}')
plt.xlabel('en yüksek kosinüs benzerliği'); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(REP / 'dedup_v2_calibration.png', dpi=110); plt.show()

**İyi bir ayrım şöyle görünür:** kırmızı solda, yeşil sağda, aralarında boşluk; τ boşluğa düşüyor ve "kesin aynı ≥ τ" yüksek (%80+). Kırmızı ile yeşil iç içeyse DINOv2 bu veride de zorlanıyor demektir; bana yaz.

### Gözle kontrol

Eşiğin hemen üstündeki ve hemen altındaki FASDD test çiftleri. Her satırda solda test görüntüsü, sağda train/val'deki en benzer komşusu.
- **Üst grid (≥ τ):** çoğu aynı sahne olmalı. Bunlar birleştirilecek.
- **Alt grid (< τ):** çoğu farklı sahne olmalı. Bunlar ayrı kalacak.

Alt grid'de hâlâ çok sayıda aynı sahne varsa `TAU`'yu elle düşür (örneğin otomatik değerden 0.03 aşağı). Üst grid'de çok sayıda farklı sahne varsa yükselt. Sonra 3. bölümü tekrar çalıştır.

In [ ]:
def show_pairs(qs, ns, ds, title, n=8):
    if len(qs) == 0:
        print('çift yok:', title); return
    pick = np.random.default_rng(SEED).choice(len(qs), min(n, len(qs)), replace=False)
    fig, axes = plt.subplots(len(pick), 2, figsize=(9, 2.6 * len(pick)), squeeze=False)
    for row, p in zip(axes, pick):
        for ax, k in [(row[0], qs[p]), (row[1], ns[p])]:
            with Image.open(WORK / m.file[k]) as im:
                im.draft('RGB', (512, 512))
                ax.imshow(im.convert('RGB'))
            ax.set_title(f'{m.split[k]} · {m.uid[k][:28]} · sim={ds[p]:.3f}', fontsize=7)
            ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

above = (d_test >= tau) & (d_test < tau + 0.03)
below = (d_test < tau) & (d_test >= tau - 0.03)
show_pairs(f_test[above], n_test[above], d_test[above], f'Eşiğin hemen ÜSTÜ (τ ≤ sim < τ+0.03): birleştirilecek')
show_pairs(f_test[below], n_test[below], d_test[below], f'Eşiğin hemen ALTI (τ-0.03 ≤ sim < τ): ayrı kalacak')

## 4. Yeniden grupla ve dağıt

Bağlantılar: 03'teki gruplar (phash ve Boreal videosu) + FASDD içinde benzerliği ≥ τ olan çiftler. Boreal v1'deki split'inde kalır (Heinola test). FASDD 03'teki kurallarla yeniden dağıtılır: %70/15/15, `BIG_GROUP`'tan büyük gruplar train.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

f_all = sel(is_f)
v, nb = topk_sim(f_all, f_all, k=K + 1)
r, c = np.nonzero(v >= tau)
a, b = f_all[r], nb[r, c]
keep = a < b
emb_pairs = np.stack([a[keep], b[keep]], axis=1)
saturated = int((v[:, -1] >= tau).sum())

memb = []
for ix in m.groupby('group').indices.values():
    memb += list(zip(ix[:-1], ix[1:]))
memb = np.array(memb, dtype=np.int64).reshape(-1, 2)

e = np.concatenate([memb, emb_pairs])
graph = coo_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(m), len(m)))
n_groups, new_group = connected_components(graph, directed=False)

sizes = pd.Series(new_group[f_all]).value_counts()
print('DINOv2 çifti (≥ τ):', len(emb_pairs), f'  · {K} komşunun hepsi eşiği geçen görüntü: {saturated}')
print('FASDD grup sayısı:', m[is_f].group.nunique(), '→', len(sizes))
print('en büyük 10 FASDD grubu:', sizes.head(10).tolist())
print(f'en büyük grup FASDD\'nin %{100 * sizes.iloc[0] / len(f_all):.1f}\'i · '
      f'{BIG_GROUP} ve altındaki gruplarda: %{100 * sizes[sizes <= BIG_GROUP].sum() / len(f_all):.1f}')
if sizes[sizes <= BIG_GROUP].sum() < 0.35 * len(f_all):
    print('UYARI: küçük gruplar %35\'in altında, %70/15/15 tutmayabilir. τ fazla düşük olabilir.')

In [ ]:
def assign(groups, ratios, forced=None, seed=SEED):
    splits = list(ratios)
    tot = groups[CATS].sum().to_numpy(float) + 1e-9
    target = np.array([[ratios[s] * t for t in tot] for s in splits])
    cur = np.zeros_like(target)
    out = {}
    forced = forced or {}
    for gid, s in forced.items():
        cur[splits.index(s)] += groups.loc[gid, CATS].to_numpy(float)
        out[gid] = s
    rest = groups.drop(index=list(forced))
    order = rest.sample(frac=1, random_state=seed)
    order = order.loc[order[CATS].sum(axis=1).sort_values(ascending=False, kind='stable').index]
    for gid, row in order.iterrows():
        v_ = row[CATS].to_numpy(float)
        cost = [(((cur[k] + v_ - target[k]) / tot) ** 2).sum() - (((cur[k] - target[k]) / tot) ** 2).sum()
                for k in range(len(splits))]
        k = int(np.argmin(cost))
        cur[k] += v_
        out[gid] = splits[k]
    return out

m2 = m.copy()
m2['old_split'] = m2.split
m2['group'] = new_group
gcat = m2.pivot_table(index='group', columns='cat', values='uid', aggfunc='count', fill_value=0)
gcat = gcat.reindex(columns=CATS, fill_value=0)
g_has_boreal = m2.groupby('group').source.agg(lambda s: (s == 'boreal').any())
g_has_test   = m2.groupby('group').event.agg(lambda s: (s == TEST_EVENT).any())
split_of_group = {g: 'test' for g in g_has_test[g_has_test].index}
# Boreal grupları bu adımda değişmiyor: v1'deki split'leri aynen korunur
b_old = m2[m2.group.isin(g_has_boreal[g_has_boreal & ~g_has_test].index)].groupby('group').old_split
assert b_old.nunique().max() == 1, 'bir Boreal grubu v1\'de iki split\'e dağılmış'
split_of_group.update(b_old.first().to_dict())
f_groups = gcat.loc[~g_has_boreal & ~g_has_test]
big = f_groups[f_groups.sum(axis=1) > BIG_GROUP].index
split_of_group.update(assign(f_groups, RATIOS, forced={g: 'train' for g in big}))
m2['split'] = m2.group.map(split_of_group)
assert m2.split.notna().all()
assert m2.groupby('group').split.nunique().max() == 1
assert (m2[m2.event == TEST_EVENT].split == 'test').all()
print('train\'e sabitlenen büyük FASDD grubu:', len(big), '  görüntü:', int(f_groups.loc[big].sum().sum()))

t = m2.pivot_table(index=['source', 'split'], columns='cat', values='uid', aggfunc='count', fill_value=0)
t = t.reindex(columns=CATS, fill_value=0)
t['toplam'] = t.sum(axis=1)
t['negatif %'] = (100 * t['negatif'] / t['toplam']).round(1)
display(t)
display(pd.crosstab(m2.old_split, m2.split, margins=True).rename_axis(index='v1', columns='v2'))

Kontrol: FASDD yine yaklaşık %70/15/15 olmalı. v1 → v2 tablosu kaç görüntünün yer değiştirdiğini gösterir.

**v1'den beklenen tablo:** FASDD 25.097 görüntü ~190 sahneye iniyor, en büyüğü ~8.000 görüntü. Zincirleme yüzünden bir sahnenin bütün videosu tek gruba toplanıyor; bu yüzden UYARI satırı çıkabilir. En büyük gruplar train'e gidince val/test'te fire+smoke oranı hedefin yarısına düşüyor ama yine ~550 görüntü kalıyor. τ gözle doğrulandıysa yükseltme: sızıntı geri gelir.

## 5. Sızıntı doğrulaması

Önce kalan eşik üstü çiftler hedefli düzeltilir (v1'de 6 test, 8 val görüntüsü kalmıştı), sonra sıfır olduğu doğrulanır.

In [ ]:
def nn_report(m_, name, q_mask, b_mask):
    q, bb = m_.index[q_mask].to_numpy(), m_.index[b_mask].to_numpy()
    d, n_ = topk_sim(q, bb)
    d, n_ = d[:, 0], n_[:, 0]
    print(f'{name}: ≥ τ {int((d >= tau).sum())} görüntü (0 olmalı) · '
          f'τ-0.03 ile τ arası {int(((d >= tau - 0.03) & (d < tau)).sum())} · medyan {np.median(d):.3f}')
    return q, n_, d

# Yoğun sahnelerde 64 komşunun hepsi kendi kümesinden çıkabiliyor; iki küme arasındaki bazı
# eşik üstü bağlantılar 4. bölümde kaçabiliyor. Kalanlar burada hedefli düzeltilir:
# sorgunun grubu, benzediği komşunun split'ine taşınır (test→train/val, val→train), 0 olana kadar.
for it in range(20):
    isf2 = m2.source == 'fasdd'
    q_t, n_t, d_t = nn_report(m2, 'FASDD test ↔ train/val', isf2 & (m2.split == 'test'), isf2 & (m2.split != 'test'))
    q_v, n_v, d_v = nn_report(m2, 'FASDD val  ↔ train    ', isf2 & (m2.split == 'val'), isf2 & (m2.split == 'train'))
    hits = [(q, n) for q, n, d in zip(q_t, n_t, d_t) if d >= tau] + \
           [(q, n) for q, n, d in zip(q_v, n_v, d_v) if d >= tau]
    if not hits:
        print(f'tur {it + 1}: ihlal yok ✓')
        break
    for q, n in hits:
        m2.loc[m2.group == m2.group[q], 'split'] = m2.split[n]
    print(f'tur {it + 1}: {len(hits)} ihlal düzeltildi\n')
assert m2.groupby('group').split.nunique().max() == 1
assert (m2[m2.event == TEST_EVENT].split == 'test').all()
print(f'karşılaştırma — v1\'de FASDD test ≥ τ: {int((d_test >= tau).sum())} görüntü')

t = m2.pivot_table(index=['source', 'split'], columns='cat', values='uid', aggfunc='count', fill_value=0)
t = t.reindex(columns=CATS, fill_value=0)
t['toplam'] = t.sum(axis=1)
t['negatif %'] = (100 * t['negatif'] / t['toplam']).round(1)
display(t)

top = np.argsort(-d_t)[:200]
show_pairs(q_t[top], n_t[top], d_t[top], 'v2: test ↔ train/val en benzer kalan çiftler (hepsi < τ)')

**Son gözle kontrol:** Bu grid, v2'de test ile train/val arasında kalan **en benzer** çiftler. Burada artık aynı sahne görmemen gerekiyor; görüyorsan `TAU`'yu düşürüp 3. bölümden itibaren tekrar çalıştır.

## 6. v2'yi yaz ve Drive'a kaydet

Dosyalar yeni split klasörlerine taşınır, klasör `firesmoke_v2` olarak yeniden adlandırılır, `data.yaml` ve test listeleri yeniden yazılır, zip Drive'a gider (~5 dk). v1 zip'i yerinde kalır.

In [ ]:
moved = m2.index[m2.split != m2.old_split]
for k in moved:
    for sub, ext in [('images', '.jpg'), ('labels', '.txt')]:
        shutil.move(str(WORK / sub / m2.old_split[k] / f'{m2.uid[k]}{ext}'),
                    str(WORK / sub / m2.split[k] / f'{m2.uid[k]}{ext}'))
for s in ['train', 'val', 'test']:
    n_img = sum(1 for _ in (WORK / 'images' / s).iterdir())
    n_lbl = sum(1 for _ in (WORK / 'labels' / s).iterdir())
    assert n_img == n_lbl == int((m2.split == s).sum()), (s, n_img, n_lbl)
print(len(moved), 'görüntü taşındı')

if DATA.exists():
    shutil.rmtree(DATA)
WORK.rename(DATA)

m2['file'] = 'images/' + m2.split + '/' + m2.uid + '.jpg'
m2['dino_nn_test'] = np.nan
m2.loc[q_t, 'dino_nn_test'] = d_t
m2.drop(columns=['old_split']).to_csv(DATA / 'manifest.csv', index=False)

def yaml_text(test='images/test'):
    return (f'path: {DATA}\ntrain: images/train\nval: images/val\ntest: {test}\n'
            'names:\n  0: fire\n  1: smoke\n')
(DATA / 'data.yaml').write_text(yaml_text())
for src in ['fasdd', 'boreal']:
    files = m2[(m2.split == 'test') & (m2.source == src)].file
    (DATA / f'test_{src}.txt').write_text('\n'.join(f'./{f}' for f in files) + '\n')
    (DATA / f'data_test_{src}.yaml').write_text(yaml_text(f'test_{src}.txt'))

summary = m2.groupby(['split', 'source']).agg(görüntü=('uid', 'size'),
                                              negatif=('cat', lambda c: int((c == 'negatif').sum())),
                                              fire_kutu=('n_fire', 'sum'), smoke_kutu=('n_smoke', 'sum'))
summary = summary.reindex(['train', 'val', 'test'], level=0)
lines = ['# Dataset v2 — firesmoke (DINOv2 ile temizlenmiş)', '',
         f'- Görüntü: {len(m2)} · grup: {n_groups} · τ: {tau:.3f} ({"otomatik, NEG_Q=" + str(NEG_Q) if TAU is None else "elle"}) · K: {K} · seed: {SEED}',
         f'- Kalibrasyon: kesin farklı ≥ τ %{100 * (d_neg >= tau).mean():.1f} · kesin aynı ≥ τ %{100 * (d_pos >= tau).mean():.1f}',
         f'- FASDD test ≥ τ: v1 {int((d_test >= tau).sum())} → v2 {int((d_t >= tau).sum())} görüntü · '
         f'val ≥ τ: {int((d_v >= tau).sum())}',
         f'- v1 → v2 yer değiştiren görüntü: {len(moved)}',
         '', '## Split × kaynak', summary.to_markdown(),
         '', '## Split × kaynak × kategori', t.to_markdown()]
report = '\n'.join(lines)
(DATA / 'README.md').write_text(report)
(REP / 'dataset_v2.md').write_text(report)
shutil.copy(DATA / 'manifest.csv', REP / 'dataset_v2_manifest.csv')

tmp = Path('/content/firesmoke_v2.zip')
if tmp.exists():
    tmp.unlink()
t0 = time.time()
subprocess.run(['zip', '-r', '-0', '-q', str(tmp), DATA.name], cwd=DATA.parent, check=True)
shutil.copy(tmp, V2_ZIP)
tmp.unlink()
print(f'{V2_ZIP}  {V2_ZIP.stat().st_size / 1e9:.2f} GB  ({time.time() - t0:.0f} sn)\n')
print(report)

**Kontrol noktası — bana gönderilecekler:**
1. 3. bölümün dört satırı (τ ve üç yüzde) ve histogramın ekran görüntüsü ya da tarifi
2. Eşiğin üstü/altı grid'leri için cevabın: üsttekiler çoğunlukla aynı sahne, alttakiler çoğunlukla farklı mı?
3. 5. bölümün iki satırı ve son grid için cevabın
4. En sondaki rapor

Bunlar tamamsa dataset kesinleşir ve `05_train` ile baseline eğitime geçeriz.